### Практическое применение данных
Мы уже собрали, отчистили, проанализировали и визуализировали полученные данные, но все еще не нашли к ним никакого практического применения. Исправим это: реализуем алгоритм поиска похожих тайтлов

загрузим данные

In [1]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

titles = pd.read_csv('datasets/titles_cleaned.tsv', sep='\t')
countries = pd.read_csv('datasets/countries_cleaned.tsv', sep='\t')
directors = pd.read_csv('datasets/directors_cleaned.tsv', sep='\t')
actors = pd.read_csv('datasets/actors_cleaned.tsv', sep='\t')
genres = pd.read_csv('datasets/genres_cleaned.tsv', sep='\t')

сопоставим каждой стране целое число, чтобы удобно представлять несколько стран-производителей одного тайтла в формате One-hot вектора

In [2]:
country_id_to_int = {c_id : i for i, c_id in enumerate(countries['country_id'].unique().tolist())}
int_to_country_id = {i : c_id for c_id, i in country_id_to_int.items()}

аналогично для жанров

In [3]:
genre_id_to_int = {g_id : i for i, g_id in enumerate(genres['genre_id'].unique().tolist())}
int_to_genre_id = {i : g_id for g_id, i in genre_id_to_int.items()}

Отберем для нашей "базы" контента только те единицы, у которых есть: хотя бы один актер, хотя бы один режиссер, хотя бы одна страна и хотя бы один жанр

Также оставим, только те признаки, по которым разумно сравнивать похожесть тайтлов

In [4]:
titles = titles[titles['title_id'].isin(set(actors['title_id'].to_list()).intersection(set(directors['title_id'].to_list())).intersection(set(countries['title_id'].to_list())).intersection(set(genres['title_id'].to_list())))]
titles = titles.drop(['date_added', 'duration', 'language', 'user_rating_size', 'user_rating_score', 'budget', 'revenue', 'rating', 'title_normalized', 'popularity'], axis=1)
titles

,type,title,release_year,description,title_id
0,Movie,Shrek Forever After,2010,A bored and domesticated Shrek pacts with deal...,t1
1,Movie,Inception,2010,"Cobb, a skilled thief who commits corporate es...",t2
2,Movie,Harry Potter and the Deathly Hallows: Part 1,2010,"Harry, Ron and Hermione walk away from their l...",t3
3,Movie,Tangled,2010,"Feisty teenager Rapunzel, who has long and mag...",t4
4,Movie,How to Train Your Dragon,2010,As the son of a Viking leader on the cusp of m...,t5
...,...,...,...,...,...
32108,Movie,Zed Plus,2014,A philandering small-town mechanic's political...,t41372
32109,Movie,Zenda,2009,A change in the leadership of a political part...,t41373
32110,Movie,Zinzana,2015,Recovering alcoholic Talal wakes up inside a s...,t41374
32112,Movie,Zombieland,2009,Looking to survive in a world taken over by zo...,t41377


#### Как мы будем реализовывать поиск похожих тайтлов:  
1. С помощью модели **Qwen3-Embedding-0.6B** построим эмбеддинги для:  
    * Текстового описания - размерность 1024
    * Тайтла - размерность 1024
2. Соберем множества для каждого тайтла (с таким большим количеством уникальных актеров и режиссеров использовать OHE вектора не разумно):
    * Множество участвовавших актеров
    * Множество участвовавших режиссеров
3. Соберем OHE векторя для каждого тайтла:
    * Вектор со странами
    * Вектор с жанрами

На вход алгоритма будем принимать тайтл из нашего хранилища, после чего считать "похожесть" следующим образом:  
* Косинусная близость для: эмбеддингов описания, эмбеддингов тайтла, OHE векторов жанра и стран
* Jaccard index (отношение мощности пересечения к мощности объединения) для множеств актеров и режиссеров
<div style="text-align: center;">
  <img class="XqHOTb IGEbUc" alt="" src="https://www.gstatic.com/education/formulas2/553212783/ru/jaccard_index.svg" role="img" data-csiid="9Bq7aeCYD7fAwPAPqcuT8A4_8" data-atf="0" width="200" >
</div>  

* Экспоненциальное затухание для года: $$w = e^{-\frac{|year - year_{idx}|}{year\_scale}}$$
* Взвешенная сумма этих значений как финальный показатель "похожести"





note: если вы просто хотите проверить работу алгоритма, то не трогайте следующие три ячейки: в них строится и сохраняется преобразованный датасет, для их запуска потребуется GPU с хотя бы 6GB видеопамяти (переходите к ячейке, в которой читается датасет в формате `pickle`)

In [6]:
def get_embedded_dataset(titles, actors, directors, countries, genres, emb_model):
    actors_by_title = actors.groupby('title_id')['actor_id'].agg(set).to_dict()
    directors_by_title = directors.groupby('title_id')['director_id'].agg(set).to_dict()
    countries_by_title = countries.groupby('title_id')['country_id'].agg(list).to_dict()
    genres_by_title = genres.groupby('title_id')['genre_id'].agg(list).to_dict()

    actors_sets, directors_sets, countries_ohe_vectors, genres_ohe_vectors, years, titles_ids = [], [], [], [], [], []
    for _, row in titles.iterrows():
        actors_sets.append(actors_by_title[row['title_id']])
        directors_sets.append(directors_by_title[row['title_id']])
        years.append(row['release_year'])

        cntr_vec = np.zeros(shape=(len(int_to_country_id),))
        cntr_vec[list(map(lambda x: country_id_to_int[x], countries_by_title[row['title_id']]))] = 1
        countries_ohe_vectors.append(cntr_vec)

        genr_vec = np.zeros(shape=(len(genre_id_to_int),))
        genr_vec[list(map(lambda x: genre_id_to_int[x], genres_by_title[row['title_id']]))] = 1
        genres_ohe_vectors.append(genr_vec)

        titles_ids.append(row['title_id'])

    with torch.no_grad():
        desc_embs = emb_model.encode(titles['description'].to_list(), show_progress_bar=True, convert_to_numpy=True, batch_size=64, normalize_embeddings=True)
        title_embs = emb_model.encode(titles['title'].to_list(), show_progress_bar=True, convert_to_numpy=True, batch_size=256, normalize_embeddings=True)

    emb_df = pd.DataFrame([titles_ids, actors_sets, directors_sets, years, countries_ohe_vectors, genres_ohe_vectors, desc_embs, title_embs]).T
    emb_df.columns = ['t_id', 'actors_set', 'director_set', 'year', 'countries', 'genres', 'desc_embs', 'title_embs']

    return emb_df

In [8]:
emb_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
emb_model.to(torch.device('cuda'))

embedded_dataset = get_embedded_dataset(titles, actors, directors, countries, genres, emb_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/340 [00:00<?, ?it/s]

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

In [9]:
embedded_dataset.to_pickle('embedded.pkl')

---

Загрузите датасет, если вы не выполняли три ячейки выше

In [5]:
embedded_dataset = pd.read_pickle('datasets/embedded.pkl')
embedded_dataset

,t_id,actors_set,director_set,year,countries,genres,desc_embs,title_embs
0,t1,"{a37196, a60246, a83982, a870, a34120}",{d9232},2010,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.015136719, -0.00049972534, -0.00970459, 0.0...","[-0.00033950806, 0.042236328, -0.01159668, -0...."
1,t2,"{a45404, a63418, a19495, a90898, a41125}",{d10040},2010,"[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, ...","[0.056640625, -0.052734375, -0.011169434, 0.04...","[0.015136719, -0.04272461, -0.012451172, -0.01..."
2,t3,"{a815, a32884, a51606, a29356, a51961}",{d1703},2010,"[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.036621094, 0.05908203, -0.010620117, 0.0061...","[0.064453125, 0.03564453, -0.008728027, -0.014..."
3,t4,"{a82672, a43718, a13543, a23641, a12712}","{d1094, d8534}",2010,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.056396484, 0.08251953, -0.0063171387, 0.007...","[0.037841797, 0.0126953125, -0.014221191, -0.0..."
4,t5,"{a49416, a69996, a54125, a55550, a9478}","{d12686, d8785}",2010,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.06542969, 0.021362305, -0.0115356445, 0.053...","[0.07861328, 0.056884766, -0.010131836, -0.033..."
...,...,...,...,...,...,...,...,...
21696,t41372,"{a7895, a58805, a41668, a28988, a42831, a19271...",{d14426},2014,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, ...","[0.017578125, -0.11376953, -0.009521484, 0.045...","[0.051757812, -0.019165039, -0.01373291, -0.03..."
21697,t41373,"{a34694, a55695, a31282, a81480, a48501, a6126...",{d13291},2009,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...","[0.030639648, -0.10205078, -0.007873535, 0.028...","[0.036621094, -0.008422852, -0.014587402, -0.0..."
21698,t41374,"{a91339, a75272, a11024, a66199, a9829, a82842}",{d11203},2015,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, ...","[0.0154418945, -0.05053711, -0.00793457, 0.083...","[0.029174805, 0.004333496, -0.014770508, -0.00..."
21699,t41377,"{a20290, a32211, a25982, a24003, a23033, a4508...",{d595},2009,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","[0.05908203, -0.07861328, -0.007873535, 0.0825...","[0.026489258, -0.0018692017, -0.012573242, -0...."


In [6]:
def jaccard_sim(s1 : set, s2 : set):
    return len(s1.intersection(s2)) / len(s1.union(s2))

def get_topk_similar(store, t_id, topk=10, weights=None, year_scale=5):
    if weights is None or len(weights) == 0:
        weights = np.array([0.10, 0.40, 0.18, 0.05, 0.10, 0.12, 0.05])

    idx = store[store['t_id'] == t_id].index
    idx = idx[0]

    title_sim = np.stack(store['title_embs'].values) @ store['title_embs'][idx]
    desc_sim = np.stack(store['desc_embs'].values) @ store['desc_embs'][idx]

    genres_mat = np.stack(store['genres'].values).astype(float)
    genre_sim = (genres_mat @ genres_mat[idx]) / (np.linalg.norm(genres_mat, axis=1) * np.linalg.norm(genres_mat[idx]) + 1e-8)

    countries_mat = np.stack(store['countries'].values).astype(float)
    country_sim = (countries_mat @ countries_mat[idx]) / (np.linalg.norm(countries_mat, axis=1) * np.linalg.norm(countries_mat[idx]) + 1e-8)

    dir_sim = np.array([jaccard_sim(store['director_set'][idx], s2) for s2 in store['director_set']])
    act_sim = np.array([jaccard_sim(store['actors_set'][idx], s2) for s2 in store['actors_set']])
    year_sim = np.exp(-np.abs((store['year'] - store['year'][idx]).astype(float)) / year_scale)

    final_sim = weights @ np.array([title_sim, desc_sim, genre_sim, country_sim, dir_sim, act_sim, year_sim])

    result = store[['t_id', 'year']].copy()
    result['title_sim'] = title_sim
    result['desc_sim'] = desc_sim
    result['genre_sim'] = genre_sim
    result['country_sim'] = country_sim
    result['dir_sim'] = dir_sim
    result['act_sim'] = act_sim
    result['year_sim'] = year_sim
    result['final_sim'] = final_sim

    result = result[result['t_id'] != t_id]
    result = result.sort_values('final_sim', ascending=False).head(topk).reset_index(drop=True)

    return result

Найдем 15 наиболее похожих тайтлов на мультфильм `Shrek Forever After` - его id - **t1**

In [7]:
similar = get_topk_similar(embedded_dataset, 't1', topk=15)
titles.merge(similar, right_on='t_id', left_on='title_id', how='inner').sort_values(by='final_sim', ascending=False)

,type,title,release_year,description,title_id,t_id,year,title_sim,desc_sim,genre_sim,country_sim,dir_sim,act_sim,year_sim,final_sim
1,Movie,Scared Shrekless,2010,"Shrek challenges Donkey, Puss in Boots and his...",t53,t53,2010,0.902545,0.709339,0.632456,1.000000,0.0,0.428571,1.000000,0.639261
2,Movie,Donkey's Christmas Shrektacular,2010,Deck the halls with Donkey’s laughter in this ...,t204,t204,2010,0.762944,0.463259,0.774597,1.000000,0.0,1.000000,1.000000,0.621025
10,Movie,Shrek the Musical,2013,Put out of his swamp solitude by a wicked tyra...,t3111,t3111,2013,0.907637,0.768141,0.670820,1.000000,0.0,0.000000,0.548812,0.596208
6,Movie,Alvin and the Chipmunks: Chipwrecked,2011,"Playing around while aboard a cruise ship, the...",t1062,t1062,2011,0.654199,0.504195,0.670820,1.000000,1.0,0.000000,0.818731,0.578782
4,Movie,Puss in Boots,2011,"Long before he even met Shrek, the notorious f...",t1012,t1012,2011,0.773625,0.537122,0.894427,1.000000,0.0,0.111111,0.818731,0.557478
12,Movie,Trolls,2016,After the monstrous Bergens invade Troll Villa...,t6029,t6029,2016,0.583230,0.464730,0.800000,1.000000,1.0,0.000000,0.301194,0.553275
7,Movie,Tom and Jerry & The Wizard of Oz,2011,"After a deadly storm, Tom and Jerry find thems...",t1140,t1140,2011,0.723753,0.553280,0.894427,1.000000,0.0,0.000000,0.818731,0.545621
8,Movie,Puss in Boots: The Three Diablos,2012,Puss in Boots is on a mission to recover the P...,t2044,t2044,2012,0.731394,0.538953,0.774597,1.000000,0.0,0.250000,0.670320,0.541664
5,Movie,The Smurfs,2011,When the evil wizard Gargamel chases the tiny ...,t1040,t1040,2011,0.786086,0.495279,0.894427,1.000000,0.0,0.000000,0.818731,0.528653
11,Movie,The SpongeBob Movie: Sponge Out of Water,2015,Burger Beard is a pirate who is in search of t...,t5039,t5039,2015,0.746091,0.468779,0.894427,0.707107,0.5,0.000000,0.367879,0.526867


Отлично, в подборке можно встретить другие фильмы про Шрека и его друзей, фильм про кота в сапогах, кунгфу панду, тома и джерри, и другие мультфильмы

Посмотрим на 15 наиболее похожих фильмов с `Game of Thrones: The Last Watch` (фильм про закулисье сериала "Игра Престолов") - id **t9400**

In [8]:
similar = get_topk_similar(embedded_dataset, 't9400', topk=15)
titles.merge(similar, right_on='t_id', left_on='title_id', how='inner').sort_values(by='final_sim', ascending=False)

,type,title,release_year,description,title_id,t_id,year,title_sim,desc_sim,genre_sim,country_sim,dir_sim,act_sim,year_sim,final_sim
12,Movie,Making of The Last of Us,2023,Featuring extensive interviews with the cast a...,t13932,t13932,2023,0.696775,0.608743,1.0,1.0,0.0,0.000000,0.449329,0.565641
8,Movie,Marvel Studios Assembled: The Making of Eternals,2022,Join director Chloe Zhao and the Cast of Etern...,t12599,t12599,2022,0.675438,0.509890,1.0,1.0,0.0,0.111111,0.548812,0.542274
3,Movie,One Crazy Summer: A Look Back at Gravity Falls,2018,A look behind-the-scenes at the making of Grav...,t8961,t8961,2018,0.579266,0.512792,1.0,1.0,0.0,0.000000,0.818731,0.533980
14,Movie,They’ll Love Me When I’m Dead,2018,"Actors, crew members and others who were there...",t38404,t38404,2018,0.620348,0.500943,1.0,1.0,0.0,0.000000,0.818731,0.533348
9,Movie,Marvel Studios Assembled: The Making of Thor: ...,2022,"Settle in with the likes of Taika Waititi, Chr...",t12699,t12699,2022,0.679887,0.519585,1.0,1.0,0.0,0.000000,0.548812,0.533263
2,Movie,What We Left Behind: Looking Back at Star Trek...,2018,A documentary exploring the legacy of Star Tre...,t8800,t8800,2018,0.648429,0.489224,1.0,1.0,0.0,0.000000,0.818731,0.531469
4,Movie,The Summers of It – Chapter Two: It Ends,2019,This documentary focuses on the actors and the...,t9454,t9454,2019,0.606028,0.462975,1.0,1.0,0.0,0.000000,1.000000,0.525793
11,Movie,Timeless Heroes: Indiana Jones & Harrison Ford,2023,An in-depth look at an incredible moment in fi...,t13765,t13765,2023,0.670046,0.509296,1.0,1.0,0.0,0.000000,0.449329,0.523190
1,Movie,Grounded: Making The Last of Us,2013,A feature-length exploration of the game's cre...,t3645,t3645,2013,0.669574,0.525248,1.0,1.0,0.0,0.000000,0.301194,0.522116
10,Movie,Marvel Studios Assembled: The Making of the Gu...,2023,Join visionary director James Gunn and superst...,t13644,t13644,2023,0.607690,0.517455,1.0,1.0,0.0,0.000000,0.449329,0.520218


В подборке встречаются фильмы, в названии которых есть "Making": мы проверили, действительно, большинстов из них это формат "фильм о фильме"

Найдем кино, которое будет похоже на `Iron Man 2` - id - `t8`

In [9]:
similar = get_topk_similar(embedded_dataset, 't8', topk=15)
titles.merge(similar, right_on='t_id', left_on='title_id', how='inner').sort_values(by='final_sim', ascending=False)

,type,title,release_year,description,title_id,t_id,year,title_sim,desc_sim,genre_sim,country_sim,dir_sim,act_sim,year_sim,final_sim
3,Movie,Iron Man 3,2013,When Tony Stark's world is torn apart by a for...,t3008,t3008,2013,0.962767,0.704935,1.000000,1.000000,0.0,0.428571,0.548812,0.687120
6,Movie,Avengers: Age of Ultron,2015,When Tony Stark tries to jumpstart a dormant p...,t5000,t5000,2015,0.866671,0.653375,1.000000,1.000000,0.0,0.250000,0.367879,0.626411
2,Movie,The Avengers,2012,When an unexpected enemy emerges and threatens...,t1998,t1998,2012,0.837839,0.596991,1.000000,1.000000,0.0,0.250000,0.670320,0.616096
8,Movie,Captain America: Civil War,2016,"Following the events of Age of Ultron, the col...",t5995,t5995,2016,0.813195,0.634932,1.000000,1.000000,0.0,0.250000,0.301194,0.610352
0,Movie,Captain America: The First Avenger,2011,"During World War II, Steve Rogers is a sickly ...",t1006,t1006,2011,0.855101,0.616492,1.000000,1.000000,0.0,0.000000,0.818731,0.603043
13,Movie,Marvel's Iron Man & Hulk: Heroes United,2013,When scientists join Hulk's gamma energy with ...,t40290,t40290,2013,0.861594,0.629478,1.000000,1.000000,0.0,0.000000,0.548812,0.595391
4,Movie,Captain America: The Winter Soldier,2014,After the cataclysmic events in New York with ...,t4002,t4002,2014,0.824158,0.598914,1.000000,1.000000,0.0,0.111111,0.449329,0.587781
11,Movie,Avengers: Endgame,2019,After the devastating events of Avengers: Infi...,t8989,t8989,2019,0.873951,0.547245,1.000000,1.000000,0.0,0.250000,0.165299,0.574558
10,Movie,Avengers: Infinity War,2018,As the Avengers and their allies have continue...,t7990,t7990,2018,0.876093,0.582642,1.000000,1.000000,0.0,0.111111,0.201897,0.574094
5,Movie,The Amazing Spider-Man 2,2014,"For Peter Parker, life is busy. Between taking...",t4006,t4006,2014,0.884541,0.565024,1.000000,1.000000,0.0,0.000000,0.449329,0.566930


Фильмы в подборке действительно похожи - сиквелы и приквелы Железного человека и другие фильмы о супергероях

Найдем похожие фильмы на `Transformers: Dark of the Moon` - его id - **t1167**

In [10]:
similar = get_topk_similar(embedded_dataset, 't1167', topk=15)
titles.merge(similar, right_on='t_id', left_on='title_id', how='inner').sort_values(by='final_sim', ascending=False)

,type,title,release_year,description,title_id,t_id,year,title_sim,desc_sim,genre_sim,country_sim,dir_sim,act_sim,year_sim,final_sim
8,Movie,Transformers: Age of Extinction,2014,As humanity picks up the pieces after the batt...,t4007,t4007,2014,0.906173,0.654209,1.000000,1.0,1.0,0.111111,0.548812,0.723075
10,Movie,Transformers: The Last Knight,2017,Humans and Transformers are at war. Optimus Pr...,t7014,t7014,2017,0.916373,0.605206,1.000000,1.0,1.0,0.111111,0.301194,0.692113
12,Movie,Transformers: Rise of the Beasts,2023,When a new threat capable of destroying the en...,t12985,t12985,2023,0.925200,0.594664,1.000000,1.0,0.0,0.111111,0.090718,0.578255
2,Movie,Biohazard: Patient Zero,2011,Two young scientists are swept up in a governm...,t1995,t1995,2011,0.685719,0.512644,1.000000,1.0,0.0,0.000000,1.000000,0.553630
13,Movie,Transformers One,2024,The untold origin story of Optimus Prime and M...,t14017,t14017,2024,0.937565,0.623344,0.816497,1.0,0.0,0.000000,0.074274,0.543777
0,Movie,Iron Man 2,2010,With the world now aware of his dual life as t...,t8,t8,2010,0.745194,0.488931,1.000000,1.0,0.0,0.000000,0.818731,0.541028
6,Movie,Transformers Prime: Beast Hunters - Predacons ...,2013,"Autobots, Decepticons, Predaking and Predacons...",t3250,t3250,2013,0.868911,0.558995,0.707107,1.0,0.0,0.111111,0.670320,0.534618
5,Movie,After Earth,2013,One thousand years after cataclysmic events fo...,t3096,t3096,2013,0.751221,0.474854,1.000000,1.0,0.0,0.000000,0.670320,0.528580
4,Movie,The Amazing Spider-Man,2012,Peter Parker is an outcast high schooler aband...,t2003,t2003,2012,0.748173,0.455518,1.000000,1.0,0.0,0.000000,0.818731,0.527961
14,Movie,The Matrix Revolutions,2003,The final installment in the Matrix trilogy fi...,t41071,t41071,2003,0.744192,0.521501,1.000000,1.0,0.0,0.000000,0.201897,0.523114


Действительно похожие фильмы

Найдем фильмы похожие на 28 Панфиловцев, его id - t6345

In [11]:
similar = get_topk_similar(embedded_dataset, 't6345', topk=15)
titles.merge(similar, right_on='t_id', left_on='title_id', how='inner').sort_values(by='final_sim', ascending=False)

,type,title,release_year,description,title_id,t_id,year,title_sim,desc_sim,genre_sim,country_sim,dir_sim,act_sim,year_sim,final_sim
6,Movie,T-34,2018,"In 1944, a courageous group of Russian soldier...",t8330,t8330,2018,0.665045,0.555481,1.000000,1.000000,0.0,0.0,0.670320,0.552213
11,Movie,The Last Frontier,2020,The story of the Podolsk cadets’ heroic stand ...,t10199,t10199,2020,0.746084,0.583167,0.866025,1.000000,0.0,0.0,0.449329,0.536226
14,Movie,The Pilot: A Battle for Survival,2021,"December of 1941, Northwestern Front. A German...",t11753,t11753,2021,0.729994,0.593474,0.866025,1.000000,0.0,0.0,0.367879,0.534668
0,Movie,Fortress of War,2010,The film covers the heroic defence of the Bres...,t391,t391,2010,0.701600,0.562396,1.000000,0.707107,0.0,0.0,0.301194,0.525534
10,Movie,Kalashnikov AK-47,2020,Tank commander Kalashnikov is severely injured...,t10167,t10167,2020,0.688210,0.535391,0.866025,1.000000,0.0,0.0,0.449329,0.511328
13,Movie,Red Ghost,2021,"December 30, 1941. Vyazma ('Vyazemsky cauldron...",t11735,t11735,2021,0.697495,0.537464,0.866025,1.000000,0.0,0.0,0.367879,0.509014
8,Movie,Leaving Afghanistan,2019,1988-1989. The end of the Soviet-Afghan war. T...,t9596,t9596,2019,0.621982,0.516201,0.866025,1.000000,0.0,0.0,0.548812,0.502004
4,Movie,The Dawns Here Are Quiet,2015,"It is late spring of 1942, and the Great Patri...",t5707,t5707,2015,0.661379,0.521378,0.707107,1.000000,0.0,0.0,0.818731,0.492905
5,Movie,The Icebreaker,2016,The story is based on the real events of 1985....,t6543,t6543,2016,0.723744,0.483056,0.707107,1.000000,0.0,0.0,1.000000,0.492876
3,Movie,White Tiger,2012,"Great Patriotic War, 1945. After barely surviv...",t2352,t2352,2012,0.705302,0.604048,0.577350,1.000000,0.0,0.0,0.449329,0.488539


В подборке много похожих фильмов: военные фильмы про СССР